# 01 Attention Shapes

这个 notebook 不要求你一次看懂完整 attention。它只解决一个问题：**Transformer 里一串 token 的向量，如何被拆成多个 attention head，又如何合并回原来的形状。**

先把它当成一个 shape 调试器，而不是数学课。


## 先知道你在干嘛

你会看到这条主线：

```text
x:      (batch, seq, dim)
q/k/v:  (batch, heads, seq, head_dim)
scores: (batch, heads, seq, seq)
output: (batch, heads, seq, head_dim)
merged: (batch, seq, dim)
```

学习时先别盯随机数。只盯三件事：

1. `seq` 代表 token 位置数。
2. `heads` 代表把同一个 token 向量分成几组并行视角。
3. `scores` 的最后两个维度是 `(seq, seq)`，意思是每个 token 都在给每个 token 打分。


## 使用方式

每一节都按这个节奏来：

1. 先读“本节目标”。
2. 运行代码 cell。
3. 只回答当前小问题，不急着理解后面的内容。

如果你运行完还是迷糊，就回到顶部那条 shape 主线，把当前输出填进去。


## 0. 导入工具

本节目标：准备 PyTorch 和画图工具。这里还没有 Transformer 概念。


In [3]:
import math

import matplotlib.pyplot as plt
import torch

torch.manual_seed(7)
torch.set_printoptions(precision=3, sci_mode=False)

## 1. 准备一个假的输入 `x`

本节目标：构造一个很小的 batch，模拟 embedding 之后的 token 向量。

这里的 `x` 不是原始文本，而是模型已经拿到的向量表示：

- `batch = 2`：一次看 2 条样本。
- `seq = 4`：每条样本有 4 个 token 位置。
- `dim = 8`：每个 token 用 8 维向量表示。

运行后只看 `x.shape`。


In [4]:
batch, seq, dim, heads = 2, 4, 8, 2
head_dim = dim // heads

x = torch.randn(batch, seq, dim)

print("batch:", batch)
print("seq:", seq)
print("dim:", dim)
print("heads:", heads)
print("head_dim:", head_dim)
print("x.shape:", tuple(x.shape))

batch: 2
seq: 4
dim: 8
heads: 2
head_dim: 4
x.shape: (2, 4, 8)


检查点：如果输出是 `x.shape: (2, 4, 8)`，它的意思不是 2 行 4 列表格，而是：

```text
2 个样本 × 每个样本 4 个 token × 每个 token 8 个数字
```


## 2. 一次线性层生成 q/k/v

本节目标：理解 `q`、`k`、`v` 的来源。

先不用纠结 query/key/value 的语义。工程上可以先这样理解：同一个 `x` 经过线性层后，被投影成三份向量，后面 attention 会用这三份向量来算“谁该看谁”。

运行后只确认：切分前后，`q`、`k`、`v` 的形状都还是 `(batch, seq, dim)`。


In [5]:
qkv = torch.nn.Linear(dim, dim * 3, bias=False)
q, k, v = qkv(x).chunk(3, dim=-1)

print("q before split:", tuple(q.shape))
print("k before split:", tuple(k.shape))
print("v before split:", tuple(v.shape))

q before split: (2, 4, 8)
k before split: (2, 4, 8)
v before split: (2, 4, 8)


## 3. 拆成多个 attention head

本节目标：看清 `dim` 如何被分给多个 head。

现在 `dim = 8`，`heads = 2`，所以每个 head 只拿 `head_dim = 4` 维。

```text
原来：每个 token 8 维
拆后：2 个 head × 每个 head 4 维
```

运行后重点看：`q` 从 `(2, 4, 8)` 变成 `(2, 2, 4, 4)`。


In [ ]:
def split_heads(t):
    return t.view(batch, seq, heads, head_dim).transpose(1, 2)


q = split_heads(q)
k = split_heads(k)
v = split_heads(v)

print("q after split:", tuple(q.shape))
print("k after split:", tuple(k.shape))
print("v after split:", tuple(v.shape))

检查点：`(2, 2, 4, 4)` 按顺序读作：

```text
batch=2, heads=2, seq=4, head_dim=4
```

这一步最容易迷糊，因为维度顺序从 `(batch, seq, dim)` 变成了 `(batch, heads, seq, head_dim)`。


## 4. 计算 attention

本节目标：看懂 `scores` 为什么是 `(batch, heads, seq, seq)`。

对每个 batch、每个 head，attention 都会做一张 `seq × seq` 的打分表：

- 行：当前正在更新的 token 位置，也叫 query position。
- 列：它可以读取的信息来源位置，也叫 key position。

所以 `seq = 4` 时，每个 head 都有一张 `4 × 4` 的表。


In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    output = weights @ v
    return output, weights, scores


output, weights, scores = scaled_dot_product_attention(q, k, v)
merged = output.transpose(1, 2).contiguous().view(batch, seq, dim)

shape_rows = [
    ("x", x.shape),
    ("q/k/v", q.shape),
    ("attention scores", scores.shape),
    ("attention weights", weights.shape),
    ("output per head", output.shape),
    ("merged output", merged.shape),
]

for name, shape in shape_rows:
    print(f"{name:18s} {tuple(shape)}")

## 5. 检查 attention weights 是概率分布

本节目标：确认 softmax 之后，每一行的权重加起来等于 1。

这表示：对某个 query token 来说，它会把 100% 的注意力分配到所有 key token 上。


In [ ]:

row_sums = weights[0, 0].sum(dim=-1)

print("batch 0, head 0 的每一行 attention 权重和：")
print(row_sums)


如果每个值都接近 `1.0`，说明这一行确实是一组概率。

这时你可以把 attention weights 理解成：**当前位置在读前后文时，对每个位置分配了多少注意力。**


## 6. 可视化一个 head

本节目标：把一张 `seq × seq` 的 attention 表画出来。

这里只看第 0 个 batch、第 0 个 head。横轴是被看的 token 位置，纵轴是当前 token 位置。颜色越亮，表示权重越大。


In [ ]:
one_head = weights[0, 0].detach()

plt.figure(figsize=(5, 4))
plt.imshow(one_head, cmap="viridis")
plt.colorbar(label="attention weight")
plt.xlabel("key position")
plt.ylabel("query position")
plt.title("Attention weights: batch 0, head 0")
plt.show()

one_head

## 这一页真正要带走什么

如果你今天只记住三句话，就够了：

1. Transformer block 输入和输出通常都保持 `(batch, seq, dim)`，这样后面才能接 residual connection。
2. 多头 attention 会把 `dim` 拆成 `heads × head_dim`，并行学习不同的读取方式。
3. `scores` / `weights` 是 `(batch, heads, seq, seq)`，最后的 `seq × seq` 表示“每个 token 对每个 token 的关注”。


## 练习：只改一个参数

不要一次改很多东西。按顺序试：

1. 把 `seq` 从 `4` 改成 `6`，重新运行全部 cell，观察 `scores` 是否变成 `(2, 2, 6, 6)`。
2. 把 `heads` 从 `2` 改成 `4`，保持 `dim = 8`，观察 `head_dim` 是否变成 `2`。
3. 故意把 `heads` 改成 `3`，看看哪里会报错。这个错误能帮你理解为什么 `dim` 必须能被 `heads` 整除。

每次只回答一个问题：**哪个维度变了，为什么？**
